<a href="https://colab.research.google.com/github/Ankjy/build-your-own-x/blob/master/Agence%20la%20plus%20proche.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Bonne idée ! Pour tenir compte du réseau routier, on va utiliser **OSRM** (Open Source Routing Machine) qui est gratuit, sans clé API, et s'appuie sur les données OSM.

# ```python
# ============================================================
# INSTALLATION & IMPORTS
# ============================================================
!pip install pandas numpy openpyxl requests geopy folium -q

import pandas as pd
import numpy as np
import math
import time
import requests
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
from google.colab import files
import folium
from folium.plugins import MarkerCluster
import json

print("✅ Librairies chargées")
# ```

# ```python
# ============================================================
# 1. TÉLÉVERSEMENT DES FICHIERS
# ============================================================

print("📂 Téléversez vos deux fichiers (villes sans agence + réseau)")
uploaded = files.upload()

file_names = list(uploaded.keys())
print(f"\n✅ Fichiers reçus : {file_names}")
# ```

# ```python
# ============================================================
# 2. CHARGEMENT DES FICHIERS
# ============================================================

print("\n📋 Fichiers disponibles :")
for i, f in enumerate(file_names):
    print(f"  [{i}] {f}")

idx_villes = int(input("\nEntrez l'index du fichier VILLES SANS AGENCE : "))
idx_reseau = int(input("Entrez l'index du fichier RÉSEAU (agences)    : "))

def load_file(fname):
    if fname.endswith(".xlsx") or fname.endswith(".xls"):
        return pd.read_excel(fname)
    elif fname.endswith(".csv"):
        return pd.read_csv(fname, sep=None, engine="python")
    else:
        raise ValueError(f"Format non supporté : {fname}")

df_villes = load_file(file_names[idx_villes])
df_reseau = load_file(file_names[idx_reseau])

print(f"\n✅ Villes sans agence : {df_villes.shape[0]} lignes")
print(f"✅ Réseau (agences)   : {df_reseau.shape[0]} lignes")

display(df_villes.head(3))
display(df_reseau.head(3))
# ```

# ```python
# ============================================================
# 3. IDENTIFICATION DES COLONNES CLÉS
# ============================================================

print("\n📋 Colonnes du fichier VILLES SANS AGENCE :")
for i, c in enumerate(df_villes.columns):
    print(f"  [{i}] {c}")

col_ville_nom = input("\nNom de la colonne 'nom de la ville' : ").strip()

has_lat = "lat" in df_villes.columns
has_lon = "lon" in df_villes.columns

if has_lat and has_lon:
    n_missing = df_villes[["lat","lon"]].isnull().any(axis=1).sum()
    print(f"✅ Colonnes lat/lon présentes — {n_missing} villes à géocoder")
else:
    print("⚠️  Pas de colonnes lat/lon → création par géocodage")
    df_villes["lat"] = np.nan
    df_villes["lon"] = np.nan

print("\n📋 Colonnes du fichier RÉSEAU :")
for i, c in enumerate(df_reseau.columns):
    print(f"  [{i}] {c}")

col_agence_nom = input("\nNom de la colonne 'nom agence' : ").strip()
col_agence_lat = input("Nom de la colonne 'lat'         : ").strip()
col_agence_lon = input("Nom de la colonne 'lon'         : ").strip()

print("\n✅ Configuration terminée")
# ```

# ```python
# ============================================================
# 4. GÉOCODAGE DES VILLES SANS COORDONNÉES (Nominatim OSM)
# ============================================================

geolocator = Nominatim(user_agent="agence_proximity_routing_v1", timeout=10)

def geocode_ville(nom_ville, country="Maroc", retries=3):
    query = f"{nom_ville}, {country}"
    for attempt in range(retries):
        try:
            location = geolocator.geocode(query, exactly_one=True)
            if location:
                return round(location.latitude, 6), round(location.longitude, 6)
            else:
                location = geolocator.geocode(nom_ville, exactly_one=True)
                if location:
                    return round(location.latitude, 6), round(location.longitude, 6)
                return None, None
        except GeocoderTimedOut:
            print(f"  ⏱️  Timeout '{nom_ville}' — tentative {attempt+1}/{retries}")
            time.sleep(2)
        except GeocoderServiceError as e:
            print(f"  ❌ Erreur service '{nom_ville}' : {e}")
            time.sleep(3)
    return None, None


mask_missing = df_villes["lat"].isnull() | df_villes["lon"].isnull()
idx_to_geocode = df_villes[mask_missing].index

print(f"\n🌍 Géocodage de {len(idx_to_geocode)} villes...\n")

success_count = 0
fail_list = []

for i, idx in enumerate(idx_to_geocode):
    ville_name = df_villes.loc[idx, col_ville_nom]
    lat, lon = geocode_ville(ville_name)

    if lat is not None:
        df_villes.loc[idx, "lat"] = lat
        df_villes.loc[idx, "lon"] = lon
        success_count += 1
        print(f"  [{i+1:3d}/{len(idx_to_geocode)}] ✅ {ville_name} → {lat}, {lon}")
    else:
        fail_list.append(ville_name)
        print(f"  [{i+1:3d}/{len(idx_to_geocode)}] ❌ {ville_name} → non trouvé")

    time.sleep(1)

print(f"\n📊 Géocodage : {success_count} succès / {len(fail_list)} échecs")
if fail_list:
    print("⚠️  Villes non géocodées :", fail_list)
# ```

# ```python
# ============================================================
# 5. FONCTIONS OSRM — DISTANCE & DURÉE ROUTIÈRE
# ============================================================

OSRM_BASE_URL = "http://router.project-osrm.org"

def osrm_route(lat1, lon1, lat2, lon2, retries=3):
    """
    Interroge l'API OSRM pour obtenir :
    - distance routière en km
    - durée en minutes
    entre deux points.
    Retourne (distance_km, duree_min) ou (None, None) si échec.
    """
    url = (
        f"{OSRM_BASE_URL}/route/v1/driving/"
        f"{lon1},{lat1};{lon2},{lat2}"
        f"?overview=false&annotations=false"
    )

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                if data.get("code") == "Ok":
                    route = data["routes"][0]
                    dist_km  = round(route["distance"] / 1000, 2)
                    duree_min = round(route["duration"] / 60, 1)
                    return dist_km, duree_min
                else:
                    return None, None
            else:
                time.sleep(1)
        except requests.exceptions.RequestException as e:
            print(f"  ⚠️  OSRM erreur ({attempt+1}/{retries}) : {e}")
            time.sleep(2)

    return None, None


def osrm_table(sources, destinations, retries=3):
    """
    Utilise l'API OSRM /table pour calculer en UNE SEULE requête
    toutes les distances entre N sources et M destinations.

    sources      : liste de tuples (lat, lon)
    destinations : liste de tuples (lat, lon)

    Retourne une matrice numpy (N x M) de distances en km,
    et une matrice (N x M) de durées en minutes.
    """
    # Format OSRM : lon,lat
    coords_src  = ";".join([f"{lon},{lat}" for lat, lon in sources])
    coords_dst  = ";".join([f"{lon},{lat}" for lat, lon in destinations])
    all_coords  = ";".join([f"{lon},{lat}" for lat, lon in sources + destinations])

    src_indices = ";".join([str(i) for i in range(len(sources))])
    dst_indices = ";".join([str(i) for i in range(len(sources), len(sources) + len(destinations))])

    url = (
        f"{OSRM_BASE_URL}/table/v1/driving/{all_coords}"
        f"?sources={src_indices}&destinations={dst_indices}"
        f"&annotations=distance,duration"
    )

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200:
                data = resp.json()
                if data.get("code") == "Ok":
                    dist_matrix = np.array(data["distances"]) / 1000   # → km
                    dur_matrix  = np.array(data["durations"]) / 60     # → minutes
                    return np.round(dist_matrix, 2), np.round(dur_matrix, 1)
            time.sleep(2)
        except requests.exceptions.RequestException as e:
            print(f"  ⚠️  OSRM Table erreur ({attempt+1}/{retries}) : {e}")
            time.sleep(3)

    return None, None


def haversine_fallback(lat1, lon1, lat2, lon2):
    """Distance à vol d'oiseau (fallback si OSRM échoue)."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return round(R * 2 * math.asin(math.sqrt(a)), 2)
# ```

# ```python
# ============================================================
# 6. CALCUL DES DISTANCES ROUTIÈRES — STRATÉGIE PAR BATCH
# ============================================================
# On utilise l'API /table d'OSRM qui calcule une matrice
# complète en une seule requête → beaucoup plus rapide !
# OSRM public limite à ~100 coordonnées par requête.
# On découpe donc en batches de 50 villes x toutes agences.
# ============================================================

# Nettoyage réseau
df_reseau_clean = df_reseau.dropna(subset=[col_agence_lat, col_agence_lon]).copy()
df_reseau_clean[col_agence_lat] = pd.to_numeric(df_reseau_clean[col_agence_lat], errors="coerce")
df_reseau_clean[col_agence_lon] = pd.to_numeric(df_reseau_clean[col_agence_lon], errors="coerce")
df_reseau_clean = df_reseau_clean.dropna(subset=[col_agence_lat, col_agence_lon]).reset_index(drop=True)

# Villes géocodées uniquement
df_villes_ok = df_villes.dropna(subset=["lat","lon"]).copy().reset_index(drop=True)

print(f"✅ {len(df_villes_ok)} villes avec coordonnées")
print(f"✅ {len(df_reseau_clean)} agences avec coordonnées")

# Coordonnées sous forme de listes
agences_coords = list(zip(
    df_reseau_clean[col_agence_lat],
    df_reseau_clean[col_agence_lon]
))
agences_noms = list(df_reseau_clean[col_agence_nom])

villes_coords = list(zip(df_villes_ok["lat"], df_villes_ok["lon"]))

# ----- Paramètres batch -----
BATCH_SIZE  = 50   # max villes par requête OSRM /table
n_villes    = len(villes_coords)
n_agences   = len(agences_coords)

# Matrices résultats finales
dist_matrix_full = np.full((n_villes, n_agences), np.nan)
dur_matrix_full  = np.full((n_villes, n_agences), np.nan)

print(f"\n🚗 Calcul des distances routières (OSRM /table) ...")
print(f"   {n_villes} villes × {n_agences} agences")
print(f"   Batch size : {BATCH_SIZE} villes\n")

# Vérification taille totale requête (sources + destinations ≤ 100)
if n_agences > 50:
    print(f"⚠️  {n_agences} agences > 50 → on utilisera des requêtes /route individuelles")
    USE_TABLE = False
else:
    USE_TABLE = True

# ---- Mode TABLE (rapide) ----
if USE_TABLE:
    for batch_start in range(0, n_villes, BATCH_SIZE):
        batch_end   = min(batch_start + BATCH_SIZE, n_villes)
        batch_villes = villes_coords[batch_start:batch_end]

        print(f"  🔄 Batch villes [{batch_start+1} → {batch_end}]...")

        dist_mat, dur_mat = osrm_table(batch_villes, agences_coords)

        if dist_mat is not None:
            dist_matrix_full[batch_start:batch_end, :] = dist_mat
            dur_matrix_full[batch_start:batch_end, :]  = dur_mat
            print(f"     ✅ OK")
        else:
            # Fallback Haversine pour ce batch
            print(f"     ⚠️  OSRM échoué → fallback Haversine pour ce batch")
            for i, (vlat, vlon) in enumerate(batch_villes):
                for j, (alat, alon) in enumerate(agences_coords):
                    dist_matrix_full[batch_start + i, j] = haversine_fallback(vlat, vlon, alat, alon)
                    dur_matrix_full[batch_start + i, j]  = np.nan

        time.sleep(1)

# ---- Mode ROUTE individuel (si trop d'agences) ----
else:
    total = n_villes * n_agences
    count = 0
    for i, (vlat, vlon) in enumerate(villes_coords):
        for j, (alat, alon) in enumerate(agences_coords):
            dist, dur = osrm_route(vlat, vlon, alat, alon)
            if dist is not None:
                dist_matrix_full[i, j] = dist
                dur_matrix_full[i, j]  = dur
            else:
                dist_matrix_full[i, j] = haversine_fallback(vlat, vlon, alat, alon)
                dur_matrix_full[i, j]  = np.nan
            count += 1
            if count % 50 == 0:
                print(f"  🔄 {count}/{total} paires calculées...")
            time.sleep(0.2)

print("\n✅ Matrice de distances routières complète !")
# ```

# ```python
# ============================================================
# 7. EXTRACTION DE L'AGENCE LA PLUS PROCHE (par distance route)
# ============================================================

results = []

for i, ville_row in df_villes_ok.iterrows():

    dist_row = dist_matrix_full[i]
    dur_row  = dur_matrix_full[i]

    # Index de l'agence la plus proche
    idx_closest = np.nanargmin(dist_row)

    # Fallback haversine pour les NaN
    has_fallback = np.isnan(dist_matrix_full[i, idx_closest])

    results.append({
        "ville"                   : ville_row[col_ville_nom],
        "lat_ville"               : ville_row["lat"],
        "lon_ville"               : ville_row["lon"],
        "agence_plus_proche"      : agences_noms[idx_closest],
        "lat_agence"              : agences_coords[idx_closest][0],
        "lon_agence"              : agences_coords[idx_closest][1],
        "distance_routiere_km"    : dist_matrix_full[i, idx_closest],
        "duree_trajet_min"        : dur_matrix_full[i, idx_closest],
        "methode"                 : "Haversine (fallback)" if has_fallback else "OSRM routier"
    })

# Villes non géocodées
df_villes_ko = df_villes[df_villes["lat"].isnull() | df_villes["lon"].isnull()].copy()
for _, row in df_villes_ko.iterrows():
    results.append({
        "ville"                : row[col_ville_nom],
        "lat_ville"            : None,
        "lon_ville"            : None,
        "agence_plus_proche"   : "NON DÉTERMINÉ",
        "lat_agence"           : None,
        "lon_agence"           : None,
        "distance_routiere_km" : None,
        "duree_trajet_min"     : None,
        "methode"              : "Non géocodé"
    })

df_results = pd.DataFrame(results)

print("✅ Résultats finaux calculés !")
display(df_results.head(10))
# ```

# ```python
# ============================================================
# 8. STATISTIQUES
# ============================================================

df_ok = df_results[df_results["distance_routiere_km"].notna()]

print("\n📊 STATISTIQUES DISTANCES ROUTIÈRES :")
print(f"  Distance moyenne   : {df_ok['distance_routiere_km'].mean():.1f} km")
print(f"  Distance médiane   : {df_ok['distance_routiere_km'].median():.1f} km")
print(f"  Distance min       : {df_ok['distance_routiere_km'].min():.1f} km")
print(f"  Distance max       : {df_ok['distance_routiere_km'].max():.1f} km")

print("\n⏱️  STATISTIQUES DURÉES DE TRAJET :")
df_dur = df_ok[df_ok["duree_trajet_min"].notna()]
print(f"  Durée moyenne      : {df_dur['duree_trajet_min'].mean():.0f} min")
print(f"  Durée médiane      : {df_dur['duree_trajet_min'].median():.0f} min")
print(f"  Durée max          : {df_dur['duree_trajet_min'].max():.0f} min")

print("\n📍 TOP 10 — Villes les plus éloignées (distance routière) :")
display(df_ok.nlargest(10, "distance_routiere_km")[
    ["ville", "agence_plus_proche", "distance_routiere_km", "duree_trajet_min"]
].reset_index(drop=True))

print("\n📍 Répartition des villes par agence de rattachement :")
display(df_ok["agence_plus_proche"].value_counts()
          .rename_axis("agence")
          .reset_index(name="nb_villes_rattachées"))

print("\n📍 Méthode de calcul utilisée :")
display(df_results["methode"].value_counts())
# ```

# ```python
# ============================================================
# 9. CARTE INTERACTIVE FOLIUM
# ============================================================

center_lat = df_villes_ok["lat"].mean()
center_lon = df_villes_ok["lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")

cluster_villes  = MarkerCluster(name="Villes sans agence").add_to(m)
cluster_agences = MarkerCluster(name="Agences").add_to(m)

# --- Marqueurs agences ---
for _, ag in df_reseau_clean.iterrows():
    folium.Marker(
        location=[ag[col_agence_lat], ag[col_agence_lon]],
        popup=f"<b>🏦 {ag[col_agence_nom]}</b>",
        icon=folium.Icon(color="blue", icon="building", prefix="fa")
    ).add_to(cluster_agences)

# --- Marqueurs villes + ligne vers agence la plus proche ---
for _, row in df_results[df_results["distance_routiere_km"].notna()].iterrows():

    duree_str = (f"{row['duree_trajet_min']:.0f} min"
                 if pd.notna(row["duree_trajet_min"]) else "N/A")

    # Marqueur ville
    folium.CircleMarker(
        location=[row["lat_ville"], row["lon_ville"]],
        radius=6,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>📍 {row['ville']}</b><br>"
            f"Agence : {row['agence_plus_proche']}<br>"
            f"Distance routière : {row['distance_routiere_km']} km<br>"
            f"Durée estimée : {duree_str}",
            max_width=250
        )
    ).add_to(cluster_villes)

    # Ligne ville → agence
    folium.PolyLine(
        locations=[
            [row["lat_ville"],  row["lon_ville"]],
            [row["lat_agence"], row["lon_agence"]]
        ],
        color="gray",
        weight=1.5,
        opacity=0.5,
        tooltip=f"{row['ville']} → {row['agence_plus_proche']} ({row['distance_routiere_km']} km)"
    ).add_to(m)

folium.LayerControl().add_to(m)

map_file = "carte_villes_agences.html"
m.save(map_file)
print(f"✅ Carte sauvegardée : {map_file}")
m


✅ Librairies chargées
📂 Téléversez vos deux fichiers (villes sans agence + réseau)


Saving villes_sans_agence.xlsx to villes_sans_agence (1).xlsx
Saving Réseau_agences_banque_cible.xlsx to Réseau_agences_banque_cible.xlsx

✅ Fichiers reçus : ['villes_sans_agence (1).xlsx', 'Réseau_agences_banque_cible.xlsx']

📋 Fichiers disponibles :
  [0] villes_sans_agence (1).xlsx
  [1] Réseau_agences_banque_cible.xlsx

Entrez l'index du fichier VILLES SANS AGENCE : 0
Entrez l'index du fichier RÉSEAU (agences)    : 1

✅ Villes sans agence : 92 lignes
✅ Réseau (agences)   : 34 lignes


,N°,Ville,lat,lon
0,7,Anyama,5.4964,-4.0533
1,10,Bouaflé,6.9833,-5.7500
2,12,Divo,5.8372,-5.3572


,nom_entité,nom_entité_norm,lat,lon,adresse_brute,adresse_open_street_map,adresse_norm,commune_norm,type_entité,type_entité_norm,...,banque_norm,source_file,adresse_similarity,ADM1_PCODE,ADM1_FR,ADM2_FR,ADM2_PCODE,ADM3_FR,ADM3_PCODE,clé
0,PLATEAU CAISTAB,K01_PLATEAU CAISTAB,5.324388,-4.018260,NaN,NaN,NaN,PLATEAU,AGENCE,NaN,...,ECOBANK,NaN,NaN,CI01,District Autonome D'Abidjan,Abidjan,CI0101,CI010101,Abidjan,KPLATEAUCAISTAB|AGENCE|ECOBANK
1,BOUAKE,K02_BOUAKE,7.681888,-5.028102,NaN,NaN,NaN,BOUAKE,AGENCE,NaN,...,ECOBANK,NaN,NaN,CI11,Gbeke,Bouaké,CI1103,CI110301,Bouaké,KBOUAKE|AGENCE|ECOBANK
2,TREICHVILLE MARCHE,K03_TREICHVILLE MARCHE,5.309955,-4.014249,NaN,NaN,NaN,TREICHVILLE,AGENCE,NaN,...,ECOBANK,NaN,NaN,CI01,District Autonome D'Abidjan,Abidjan,CI0101,CI010101,Abidjan,KTREICHVILLEMARCHE|AGENCE|ECOBANK



📋 Colonnes du fichier VILLES SANS AGENCE :
  [0] N°
  [1] Ville
  [2] lat
  [3] lon

Nom de la colonne 'nom de la ville' : Ville
✅ Colonnes lat/lon présentes — 0 villes à géocoder

📋 Colonnes du fichier RÉSEAU :
  [0] nom_entité
  [1] nom_entité_norm
  [2] lat
  [3] lon
  [4] adresse_brute
  [5] adresse_open_street_map
  [6] adresse_norm
  [7] commune_norm
  [8] type_entité
  [9] type_entité_norm
  [10] banque
  [11] banque_norm
  [12] source_file
  [13] adresse_similarity
  [14] ADM1_PCODE
  [15] ADM1_FR
  [16] ADM2_FR
  [17] ADM2_PCODE
  [18] ADM3_FR
  [19] ADM3_PCODE
  [20] clé

Nom de la colonne 'nom agence' : nom_entité
Nom de la colonne 'lat'         : lat
Nom de la colonne 'lon'         : lon

✅ Configuration terminée

🌍 Géocodage de 0 villes...


📊 Géocodage : 0 succès / 0 échecs
✅ 92 villes avec coordonnées
✅ 34 agences avec coordonnées

🚗 Calcul des distances routières (OSRM /table) ...
   92 villes × 34 agences
   Batch size : 50 villes

  🔄 Batch villes [1 → 50]...
     ✅ 

...


📍 Répartition des villes par agence de rattachement :


,agence,nb_villes_rattachées
0,KORHOGO,12
1,YAMOUSSOUKRO,9
2,GAGNOA,9
3,MAN,8
4,BOUAKE,6
5,DALOA,6
6,ABENGOUROU,6
7,ADZOPE,5
8,BONDOUKOU,5
9,YOPOUGON NIANGON,4



📍 Méthode de calcul utilisée :


,count
methode,
OSRM routier,92


✅ Carte sauvegardée : carte_villes_agences.html


In [4]:
# ```

# ```python
# ============================================================
# 10. EXPORT EXCEL + TÉLÉCHARGEMENT
# ============================================================

output_file = "villes_agence_plus_proche_routier.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_results.to_excel(writer, sheet_name="Résultats",        index=False)
    df_villes.to_excel(writer,  sheet_name="Villes_geocodées", index=False)

    # Matrice complète distances
    df_matrix = pd.DataFrame(
        dist_matrix_full,
        index=df_villes_ok[col_ville_nom],
        columns=agences_noms
    )
    df_matrix.to_excel(writer, sheet_name="Matrice_distances_km")

    # Matrice durées
    df_matrix_dur = pd.DataFrame(
        dur_matrix_full,
        index=df_villes_ok[col_ville_nom],
        columns=agences_noms
    )
    df_matrix_dur.to_excel(writer, sheet_name="Matrice_durees_min")

files.download(output_file)
files.download(map_file)
print(f"✅ Export terminé : {output_file} + {map_file}")
# ```

# ---

## Ce qui change par rapport à la version précédente

# | Aspect | Avant | Maintenant |
# |---|---|---|
# | **Calcul distance** | Haversine (vol d'oiseau) | **OSRM** (réseau routier réel) |
# | **Métrique** | km à vol d'oiseau | **km routiers + durée en minutes** |
# | **Performance** | Boucle individuelle | **API `/table` batch** (1 requête = N×M distances) |
# | **Fallback** | — | Haversine si OSRM indisponible |
# | **Export** | 1 onglet | **4 onglets** (résultats, villes, matrice km, matrice durées) |
# | **Carte** | — | **Carte Folium interactive** avec lignes de rattachement |

# > **OSRM public** est gratuit, sans clé API, mais limité à ~100 coordonnées/requête — géré automatiquement par le découpage en batches.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Export terminé : villes_agence_plus_proche_routier.xlsx + carte_villes_agences.html
